In [73]:
import pandas as pd
import numpy as np

In [74]:
# Load dataset
df = pd.read_csv('datafix.csv')
df.head()

,IDJwb,IDPSJ,questions,answerKeys,answer,raw_grade,max_grade,grade,labela,label,...,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24
0,1,1,Ada empat macam karakter atau olah yang diusul...,Olah pikir = Berpikiran kritis dalam segala si...,Olah pikir = Berpikiran kritis dalam segala si...,50,50,"10,00",A,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,Ada empat macam karakter atau olah yang diusul...,Olah pikir = Berpikiran kritis dalam segala si...,KARAKTER SALING MENOLONG\nKARAKTER YANG MAU SA...,30,50,"6,00",C,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,Ada empat macam karakter atau olah yang diusul...,Olah pikir = Berpikiran kritis dalam segala si...,1. olah hati adalah karakter yang penuh rasa d...,50,50,"10,00",A,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1,Ada empat macam karakter atau olah yang diusul...,Olah pikir = Berpikiran kritis dalam segala si...,Revolusi mental\nyang berarti kita harus memil...,20,50,"4,00",D,2,...,IDPSJ 03,IDPSJ 04,IDPSJ 05,IDPSJ 06,IDPSJ 07,IDPSJ 08,IDPSJ 09,IDPSJ 10,IDPSJ 11,IDPSJ 12
4,5,1,Ada empat macam karakter atau olah yang diusul...,Olah pikir = Berpikiran kritis dalam segala si...,1. Olah raga : memiliki karakter yang kuat\n2....,50,50,"10,00",A,5,...,7,5,5,7,13,63,48,40,22,5


In [75]:
# Hapus kolom yang tidak dipakai (hanya pakai sampai kolom label)
kolom_yang_dipakai = ['IDJwb', 'IDPSJ', 'questions', 'answerKeys', 'answer', 
                        'raw_grade', 'max_grade', 'grade', 'labela', 'label']
df = df[kolom_yang_dipakai]

In [76]:
# Preprocessing Step 1: Lowercase and single space
import re
def lowercase(text):
    """
    Tahap 1: Lowercase dan single space
    - Ubah semua text menjadi lowercase
    - Hapus extra spaces (multiple spaces menjadi single space)
    - Pertahankan newline untuk pemisahan kalimat
    - Menangani non-breaking space (\xa0) dan whitespace lainnya
    """
    if pd.isna(text):
        return ""
    
    # Lowercase
    text = text.lower()
    
    # Ganti semua whitespace (termasuk \xa0, \t, dll) KECUALI newline dengan spasi biasa
    # \xa0 adalah non-breaking space yang sering muncul dari copy-paste
    text = text.replace('\xa0', ' ')  # Non-breaking space
    text = text.replace('\t', ' ')     # Tab
    text = text.replace('\r', '')      # Carriage return
    
    # Normalize multiple spaces menjadi single space (TIDAK termasuk newline)
    text = re.sub(r'[ ]+', ' ', text)
    
    # Normalize multiple newlines menjadi single newline
    text = re.sub(r'\n+', '\n', text)
    
    # Hapus spasi di awal/akhir setiap baris
    lines = text.split('\n')
    lines = [line.strip() for line in lines]
    text = '\n'.join(lines)
    
    return text

In [96]:
# Preprocessing Step 2: Sentence Tokenization menggunakan NLTK + Regex
import re
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

def sentence_tokenize(text):
    """
    Tahap 2: Sentence Tokenization menggunakan NLTK + Regex
    Alur:
    1. Pisahkan per newline terlebih dahulu
    2. Untuk setiap baris, pisahkan dulu dengan regex:
       - Nomor/huruf penanda di tengah: "1.", "1)", "a.", "a)"
         (hanya jika diikuti spasi + teks, agar angka/huruf di akhir kalimat tidak terpotong)
       - Titik koma ";"
    3. Untuk setiap potongan, jalankan NLTK sent_tokenize
    4. Bersihkan penanda di awal kalimat
    Catatan: Penomoran di-split SEBELUM NLTK agar NLTK tidak salah deteksi batas kalimat.
    """
    if not text or pd.isna(text):
        return []

    # Langkah 1: Pisahkan berdasarkan newline
    lines = text.split('\n')

    sentences = []
    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Langkah 2a: Pisahkan berdasarkan pola penomoran di tengah baris
        # \s+\S memastikan ada teks setelah penanda,
        # sehingga angka/huruf di akhir kalimat (misal "ada 4." atau "partisi d.") tidak terpotong
        sub_parts = re.split(r'\s+(?=[0-9]{1,2}[\.\)]\s+\S|[a-zA-Z][\.\)]\s+\S)', line)

        for part in sub_parts:
            # Langkah 2b: Pisahkan berdasarkan ";" (titik koma)
            semi_parts = re.split(r'\s*;\s*', part)
            for sp in semi_parts:
                sp = sp.strip()
                if not sp:
                    continue

                # Langkah 3: Jalankan NLTK pada setiap potongan
                nltk_sentences = sent_tokenize(sp, language='english')

                for ns in nltk_sentences:
                    ns = ns.strip()
                    # Strip tanda kutip di awal/akhir terlebih dahulu
                    # agar penanda angka/huruf di awal bisa terdeteksi
                    ns = re.sub(r'^["""\'\'\']+|["""\'\'\']+$', '', ns).strip()
                    # Langkah 4: Hapus penanda di awal kalimat
                    ns = re.sub(
                        r'^[0-9]{1,2}[\.\)]\s*'   # 1. atau 1)
                        r'|^[0-9]{1,2}\s+'         # 1 (angka + spasi)
                        r'|^[0-9]{1,2}$'           # 1 (angka berdiri sendiri)
                        r'|^[a-zA-Z][\.\)]\s*'     # a. atau a)
                        r'|^[a-zA-Z]$'             # a (huruf berdiri sendiri)
                        r'|^-\s*'                  # - (dash)
                        r'|^\*\s*'                 # * (bullet)
                        r'|^\.\s+',                # . (titik di awal)
                        '', ns
                    ).strip()
                    # Hapus titik di akhir kalimat
                    ns = ns.rstrip('.')
                    if ns:
                        sentences.append(ns)

    return sentences


In [78]:
# Preprocessing Step 3: Normalisasi Simbol
def normalize_symbols(sentences):
    """
    Tahap 3: Normalisasi simbol khusus
    - : dan = menjadi "adalah"
    - / menjadi "atau"
    - ->, -->, --->, dst. (satu atau lebih dash diikuti >) menjadi "kemudian"
    - > menjadi "lebih besar"
    - < menjadi "lebih kecil"
    - & menjadi "dan"
    URL/link dipertahankan utuh.
    """
    if not sentences:
        return []
    
    normalized_sentences = []
    for sentence in sentences:
        # Simpan URL sebagai placeholder tanpa tanda baca agar tidak rusak saat normalisasi simbol
        urls = re.findall(r'https?://\S+|www\.\S+', sentence)
        for idx, url in enumerate(urls):
            sentence = sentence.replace(url, f'URLTOKEN{idx}')

        # Ganti ->, -->, --->, dst. dengan "kemudian" (harus sebelum > diganti)
        sentence = re.sub(r'-+>', ' kemudian ', sentence)
        
        # Ganti : dan = dengan "adalah"
        sentence = sentence.replace(':', ' adalah ')
        sentence = sentence.replace('=', ' adalah ')
        
        # Ganti / dengan "atau"
        sentence = sentence.replace('/', ' atau ')

        # Ganti > dengan "lebih besar"
        sentence = sentence.replace('>', ' lebih besar dari ')

        # Ganti < dengan "lebih kecil"
        sentence = sentence.replace('<', ' lebih kecil dari ')

        # Ganti & dengan "dan"
        sentence = sentence.replace('&', ' dan ')

        # Kembalikan URL
        for idx, url in enumerate(urls):
            sentence = sentence.replace(f'URLTOKEN{idx}', url)
        
        # Hapus extra spaces yang mungkin muncul
        sentence = re.sub(r' +', ' ', sentence).strip()
        
        if sentence:  # Hanya tambahkan jika tidak kosong
            normalized_sentences.append(sentence)
    
    return normalized_sentences


In [79]:
# Preprocessing Step 4: Menghilangkan Tanda Baca
import string

def remove_punctuation(sentences):
    """
    Tahap 4: Menghilangkan tanda baca !"#$%&'()*+,-./:;<=>?@[]^_{|}~`
    Menghapus semua tanda baca dari kalimat-kalimat,
    kecuali URL/link yang dipertahankan utuh.
    """
    if not sentences:
        return []
    
    cleaned_sentences = []
    for sentence in sentences:
        # Simpan URL sebagai placeholder tanpa tanda baca agar tidak rusak saat translate
        urls = re.findall(r'https?://\S+|www\.\S+', sentence)
        for idx, url in enumerate(urls):
            sentence = sentence.replace(url, f'URLTOKEN{idx}')

        # Hapus semua tanda baca
        translator = str.maketrans(string.punctuation, ' ' * len(string.punctuation))
        clean_sentence = sentence.translate(translator)

        # Kembalikan URL
        for idx, url in enumerate(urls):
            clean_sentence = clean_sentence.replace(f'URLTOKEN{idx}', url)

        # Hapus extra spaces yang mungkin muncul
        clean_sentence = re.sub(r' +', ' ', clean_sentence).strip()
        
        if clean_sentence:  # Hanya tambahkan jika tidak kosong
            cleaned_sentences.append(clean_sentence)
    
    return cleaned_sentences


In [84]:
df['answerKeys_lower'] = df['answerKeys'].apply(lowercase)
df['answer_lower'] = df['answer'].apply(lowercase)

In [97]:
# Terapkan pemotongan kalimat
df['answerKeys_sentences'] = df['answerKeys_lower'].apply(sentence_tokenize)
df['answer_sentences'] = df['answer_lower'].apply(sentence_tokenize)

In [98]:
# Terapkan normalisasi simbol
df['answerKeys_no_punct'] = df['answerKeys_sentences'].apply(normalize_symbols)
df['answer_no_punct'] = df['answer_sentences'].apply(normalize_symbols)

In [99]:
# Terapkan penghapusan tanda baca
df['answerKeys_clean'] = df['answerKeys_no_punct'].apply(remove_punctuation)
df['answer_clean'] = df['answer_no_punct'].apply(remove_punctuation)

In [89]:
for i in range(len(df)):
    print(f"Data ke-{i}:")
    print('Sebelum preprocessing:')
    print(df['answer'][i])
    print('Sesudah preprocessing:')
    print(df['answer_clean'][i])
    print()

Data ke-0:
Sebelum preprocessing:
Olah pikir = Berpikiran kritis dalam segala situasi yang terjadi.

Olah hati = Melakukan suatu kegiatan berdasarkan hati nurani kita.

Olah raga = Memiliki sifat dan mental yang kuat.

Olah hati dan karsa = Kreatif dalam melakukan sesuatu.
Sesudah preprocessing:
['olah pikir adalah berpikiran kritis dalam segala situasi yang terjadi', 'olah hati adalah melakukan suatu kegiatan berdasarkan hati nurani kita', 'olah raga adalah memiliki sifat dan mental yang kuat', 'olah hati dan karsa adalah kreatif dalam melakukan sesuatu']

Data ke-1:
Sebelum preprocessing:
KARAKTER SALING MENOLONG
KARAKTER YANG MAU SALING BEKERJA SAMA
KARAKTER YANG PEDULI TERHADAP PENDERITAAN ORANG LAIN
KARAKTER YANG MEMENTINGKAN SIKAP RELA BERKORBAN
Sesudah preprocessing:
['karakter saling menolong', 'karakter yang mau saling bekerja sama', 'karakter yang peduli terhadap penderitaan orang lain', 'karakter yang mementingkan sikap rela berkorban']

Data ke-2:
Sebelum preprocessing:
1. 

In [100]:
# Simpan hasil preprocessing ke TXT untuk perbandingan
with open("hasil_nltk.txt", "w", encoding="utf-8") as f:
    for i, row in enumerate(df['answer_clean']):
        f.write(f"Data {i + 1}:\n")
        if isinstance(row, list):
            for sentence in row:
                f.write(f"  {sentence}\n")
        else:
            f.write(f"  {row}\n")
        f.write("\n")

print("✓ Hasil preprocessing disimpan ke: hasil_nltk.txt")


✓ Hasil preprocessing disimpan ke: hasil_nltk.txt


In [56]:
df = df[['IDJwb', 'IDPSJ', 'questions', 'answerKeys_clean', 'answer_clean', 
                        'raw_grade', 'max_grade', 'grade', 'labela', 'label',]]

df.head()

,IDJwb,IDPSJ,questions,answerKeys_clean,answer_clean,raw_grade,max_grade,grade,labela,label
0,1,1,Ada empat macam karakter atau olah yang diusul...,[olah pikir adalah berpikiran kritis dalam seg...,[olah pikir adalah berpikiran kritis dalam seg...,50,50,"10,00",A,5
1,2,1,Ada empat macam karakter atau olah yang diusul...,[olah pikir adalah berpikiran kritis dalam seg...,"[karakter saling menolong, karakter yang mau s...",30,50,"6,00",C,3
2,3,1,Ada empat macam karakter atau olah yang diusul...,[olah pikir adalah berpikiran kritis dalam seg...,[olah hati adalah karakter yang penuh rasa dam...,50,50,"10,00",A,5
3,4,1,Ada empat macam karakter atau olah yang diusul...,[olah pikir adalah berpikiran kritis dalam seg...,"[revolusi mental, yang berarti kita harus memi...",20,50,"4,00",D,2
4,5,1,Ada empat macam karakter atau olah yang diusul...,[olah pikir adalah berpikiran kritis dalam seg...,"[olah raga adalah memiliki karakter yang kuat,...",50,50,"10,00",A,5


In [57]:
# # Simpan hasil preprocessing ke CSV
# output_filename = 'df_preprocessed.csv'

# # Simpan ke CSV
# df.to_csv(output_filename, index=False, encoding='utf-8')

# print(f"✓ Data berhasil disimpan ke: {output_filename}")
# print(f"  - Total baris: {len(df)}")
# print(f"  - Total kolom: {len(df.columns)}")
# print(f"  - Kolom: {list(df.columns)}")